#  **Unstructured - LangChain** 



---

`(1) Env 환경변수`

In [ ]:
from dotenv import load_dotenv
load_dotenv()

`(2) 기본 라이브러리`

In [ ]:
import os
from glob import glob

from pprint import pprint
import json

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import logging

# 로깅 레벨 설정 
logging.getLogger('pdfminer').setLevel(logging.ERROR)
logging.getLogger('unstructured').setLevel(logging.ERROR)

---

## **Unstructured & Langchain**

- **Unstructured**는 다양한 유형의 파일을 로드할 수 있는 문서 로더를 제공 

- 지원되는 파일 형식에는 **텍스트**, **파워포인트**, **HTML**, **PDF**, **이미지** 등이 포함

- 로컬 환경에서 사용하기 위해서는 필요한 **시스템 종속성** 설정이 필요

- 설치: pip install langchain-unstructured

- 문서: https://python.langchain.com/docs/integrations/document_loaders/unstructured_file/

**필요한 데이터 파일:**
- 테슬라_KR.txt
- 리비안_KR_with_table.pdf
- 개인정보보호법.docx
- 주택임대차보호법.txt
- 근로기준법.pdf

**참고**: 일부 파일이 없는 경우, `data/` 폴더 내 다른 TXT, PDF, DOCX 파일로 대체 가능합니다.

### 1. **UnstructuredLoader**

- **UnstructuredLoader**는 여러 파일 형식을 처리할 수 있는 범용 로더

- 하나의 로더로 **다양한 형식**의 문서를 일관되게 처리 가능

`(1) 기본 사용법`

In [ ]:
from langchain_unstructured import UnstructuredLoader

# 단일 파일 로더
loader = UnstructuredLoader("data/테슬라_KR.txt")

# 로드
docs = loader.load()
print(f"로드된 문서 수: {len(docs)}")

# 문서 출력
pprint(docs[0].page_content)

# 문서 메타데이터 출력
pprint(docs[0].metadata)

In [ ]:
# 여러 파일 로더 
loader = UnstructuredLoader([
    "data/리비안_KR_with_table.pdf",
    "data/테슬라_KR.txt" 
])

# 로드
docs = loader.load()
print(f"로드된 문서 수: {len(docs)}")

# 문서 출력
pprint(docs[10].page_content)

# 문서 메타데이터 출력
pprint(docs[10].metadata)

`(2) 문서 로딩 방법`

- **문서 로딩**은 `load()` 메서드와 **lazy_load()** 두 가지 방식으로 가능

- `load()` 메서드는 문서의 **전체 내용**을 한 번에 메모리에 로드

- **lazy_load()** 는 문서를 **페이지 단위**로 순차적으로 처리하여 메모리를 효율적으로 사용

In [ ]:
# 일반 로딩
docs = loader.load()

# 지연 로딩(lazy loading)
for doc in loader.lazy_load():
    print(doc.page_content)

`(3) 후처리(Post Processing)`

- **post_processors** 매개변수를 통해 추출된 요소를 **후처리(정제)** 가능 

- 후처리는 문자열을 입력받아 문자열을 반환하는 **함수 리스트** 형태로 지정 

In [ ]:
from unstructured.cleaners.core import clean_extra_whitespace

# 로더 생성
loader = UnstructuredLoader(
    "data/리비안_KR_with_table.pdf",
    post_processors=[clean_extra_whitespace]
)

# 문서 출력
docs = []
for doc in loader.lazy_load():
    print(doc.page_content)
    docs.append(doc)

`(4) 이미지/테이블 처리`

In [ ]:
# 로더 생성
loader = UnstructuredLoader(
    "data/리비안_KR_with_table.pdf",

    # 파티셔닝 전략 설정
    strategy="hi_res",                  # 고해상도 파티셔닝 전략 사용
    # hi_res_model_name="yolox",          # 객체탐지 모델 지정
    infer_table_structure=True,         # 표 구조를 자동으로 추론하여 HTML 형식으로 변환
    languages=["eng", "kor"],           # 영어와 한국어 문서 처리 지원
    
    # 이미지 추출 설정
    extract_images_in_pdf=True,         # PDF 내 이미지 추출 활성화
    extract_image_block_types=[         # 추출할 요소 유형 지정
        "Image",                        # 일반 이미지
        "Table"                         # 표 이미지
    ],
    extract_image_block_output_dir="data/images/리비안_KR_with_table"  # 추출된 이미지 저장 경로

)

# 문서 출력
docs = []
for doc in loader.lazy_load():
    print(doc.page_content)
    docs.append(doc)

In [ ]:
# 메타데이터 카테고리 확인
for doc in docs:
    print(doc.metadata['category'])

In [ ]:
# Table 추출
for doc in docs:
    if "Table" in doc.metadata['category']:
        print(doc)
        pprint(doc.metadata['text_as_html'])

`(6) 청킹(Chunking)`

- **Unstructured의 청킹**은 문서 형식별 **의미 단위**로 분할하는 방식을 사용

- 기존의 **단순 텍스트 기반** 청킹과 달리 각 문서 형식에 대한 **특화된 처리**를 제공

- 청킹 결과는 **CompositeElement**, **Table**, **TableChunk** 세 가지 유형으로 반환

- `chunking_strategy="basic"`, `max_characters` 설정 사용

In [ ]:
# 기본 청킹 전략 사용
loader = UnstructuredLoader(
    "data/리비안_KR_with_table.pdf",
    strategy="fast", 
    chunking_strategy="basic",
    max_characters=1000,
)
    
docs = loader.load()

print(f"로드된 문서 수: {len(docs)}")

# 문서 출력
print(docs[0].page_content)

# 문서 메타데이터 출력
pprint(docs[0].metadata)

In [ ]:
# Table, TableChunk 확인 
for doc in docs:
    if "Table" in doc.metadata['category']:
        # doc.table 속성이 있는지 확인
        if hasattr(doc, 'table'):
            print(doc.table)
        elif 'text_as_html' in doc.metadata:
            print(doc.metadata['text_as_html'])

### 2. **파일 형식별 전용 로더**

- 주요 로더:

    - **PDF**: UnstructuredPDFLoader
    - **Word**: UnstructuredWordDocumentLoader
    - **Excel**: UnstructuredExcelLoader
    - **PowerPoint**: UnstructuredPowerPointLoader
    - **HTML**: UnstructuredHTMLLoader
    - **Email**: UnstructuredEmailLoader
    - **Image**: UnstructuredImageLoader
    - **URL**: UnstructuredURLLoader


- 주요 특징:
    1. 모든 로더는 `.load()` 메서드를 통해 문서를 로드
    2. 로드된 문서는 Document 객체 리스트로 반환
    3. Document 객체:
        - page_content: 추출된 텍스트 내용
        - metadata: 파일 정보, 경로 등 메타데이터

- 사용 시 주의사항:
    1. 각 파일 형식에 맞는 시스템 의존성이 설치되어 있어야 한다
    2. 이미지나 PDF의 경우 OCR 기능을 사용하려면 tesseract-ocr이 필요하다
    3. 로더마다 추가적인 옵션을 제공할 수 있으므로, API 문서를 참고하여 필요한 옵션을 설정하는 것이 좋다
    4. 대용량 파일의 경우 메모리 사용량에 주의해야 한다

`(1) PDF 문서 로더`

In [ ]:
from langchain_community.document_loaders import UnstructuredPDFLoader

# PDF 파일 로드
loader = UnstructuredPDFLoader("data/리비안_KR_with_table.pdf")
documents = loader.load()

print(f"로드된 문서 수: {len(documents)}")

# 문서 출력
print(documents[0].page_content)

# 문서 메타데이터 출력
pprint(documents[0].metadata)

`(2) Word 문서 로더`

In [ ]:
from langchain_community.document_loaders import UnstructuredWordDocumentLoader

# Word 파일 로드
loader = UnstructuredWordDocumentLoader("data/개인정보보호법.docx")
documents = loader.load()

print(f"로드된 문서 수: {len(documents)}")

# 문서 출력
print(documents[0].page_content)

# 문서 메타데이터 출력
pprint(documents[0].metadata)

`(3) Excel 파일 로더`

In [ ]:
from langchain_community.document_loaders import UnstructuredExcelLoader

# Excel 파일 로드
loader = UnstructuredExcelLoader("data/경기도교육청_행정구역별_학제별_학급_학생_교원_20240401.xlsx")
documents = loader.load()

print(f"로드된 문서 수: {len(documents)}")

# 문서 출력
print(documents[0].page_content)

# 문서 메타데이터 출력
pprint(documents[0].metadata)

`(4) PowerPoint 파일 로더`

In [ ]:
from langchain_community.document_loaders import UnstructuredPowerPointLoader

# PowerPoint 파일 로드 
loader = UnstructuredPowerPointLoader("data/한국철도공사_8대도시(목포)_관광 형태 분석 보고서_20220101.pptx")
documents = loader.load()

print(f"로드된 문서 수: {len(documents)}")

# 문서 출력
print(documents[0].page_content)

# 문서 메타데이터 출력
pprint(documents[0].metadata)

`(5) HTML 문서 로더`

In [ ]:
from langchain_community.document_loaders import UnstructuredHTMLLoader

# HTML 파일 로드
loader = UnstructuredHTMLLoader("data/example.html")
documents = loader.load()

print(f"로드된 문서 수: {len(documents)}")

# 문서 출력
print(documents[0].page_content)

# 문서 메타데이터 출력
pprint(documents[0].metadata)

`(6) 이메일 로더`

In [ ]:
from langchain_community.document_loaders import UnstructuredEmailLoader

# 이메일 파일 로드
loader = UnstructuredEmailLoader("data/email.eml")
documents = loader.load()

print(f"로드된 문서 수: {len(documents)}")

# 문서 출력
print(documents[0].page_content)

# 문서 메타데이터 출력
pprint(documents[0].metadata)

`(7) URL 로더`

In [ ]:
from langchain_community.document_loaders import UnstructuredURLLoader

# URL에서 콘텐츠 로드
urls = ["http://example.com"]
loader = UnstructuredURLLoader(urls=urls)
documents = loader.load()

print(f"로드된 문서 수: {len(documents)}")

# 문서 출력
print(documents[0].page_content)

# 문서 메타데이터 출력
pprint(documents[0].metadata)

---

## **[실습] UnstructuredLoader를 사용한 RAG 구현**

- **다중 형식 지원**으로 PDF, DOCX, TXT 등 다양한 **법률 문서**를 통합 처리
- **Chroma DB**를 활용하여 문서의 **벡터화**와 **효율적인 검색**이 가능
- **LCEL** 기반의 체인 구성으로 **최적화된 프로세스 흐름**을 구현
- **법률 전문가 컨텍스트**가 반영된 **특화된 프롬프트**로 정확한 응답을 생성

- **제공 문서**:
    - 개인정보보호법.docx
    - 주택임대차보호법.txt
    - 근로기준법.pdf


In [ ]:
### **[실습]**

# 여기에 코드를 작성하세요.

<details close>
<summary>💡 정답 확인</summary>

```python

from langchain_community.document_loaders import (
    UnstructuredPDFLoader,
    UnstructuredWordDocumentLoader,
)
from langchain_unstructured import UnstructuredLoader
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.vectorstores.utils import filter_complex_metadata
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

import os
from typing import List, Optional, Dict
import logging

class LegalDocumentRAG:
    def __init__(self, collection_name: str, persist_directory: str = "./chroma_db"):
        # 로깅 설정
        logging.basicConfig(level=logging.INFO)
        self.logger = logging.getLogger(__name__)
        
        # 기본 설정
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
            length_function=len,
            separators=["\n\n", "\n", ".", " ", ""]
        )
        
        # 벡터 DB 초기화
        self.vectorstore = None
        
    def _load_document(self, file_path: str) -> List[Document]:
        """파일 형식에 따라 적절한 로더를 사용하여 문서 로드"""
        self.logger.info(f"Loading document: {file_path}")     
  
        try:
            if file_path.endswith('.pdf'):
                loader = UnstructuredPDFLoader(file_path)
            elif file_path.endswith('.docx'):
                loader = UnstructuredWordDocumentLoader(file_path)
            elif file_path.endswith('.txt'):
                loader = UnstructuredLoader(file_path)
            else:
                raise ValueError(f"Unsupported file format: {file_path}")
            
            documents = loader.load()

            # 메타데이터 필터링 후 파일명 추가

            # 메타데이터 처리
            for doc in documents:
                doc.metadata = {k: v for k, v in doc.metadata.items() 
                                if isinstance(v, (str, int, float, bool))}
                doc.metadata["source"] = os.path.basename(file_path)
                
            return documents
            
        except Exception as e:
            self.logger.error(f"Error loading document {file_path}: {str(e)}")
            raise

    def load_documents(self, directory: str):
        """디렉토리의 모든 문서를 로드하고 인덱싱"""
        all_documents = []
        
        # 지원하는 파일 확장자
        supported_extensions = ('.pdf', '.docx', '.txt')
        
        # 디렉토리 내 모든 파일 처리
        for file in os.listdir(directory):
            if file.endswith(supported_extensions):
                file_path = os.path.join(directory, file)
                documents = self._load_document(file_path)
                all_documents.extend(documents)
        
        # 문서 분할
        splits = self.text_splitter.split_documents(all_documents)
        self.logger.info(f"Created {len(splits)} document chunks")

        # Chroma DB에 저장
        self.vectorstore = Chroma.from_documents(
            documents=splits,
            embedding=self.embeddings,
            collection_name=self.collection_name,   
            persist_directory=self.persist_directory
        )
        
        return len(splits)

    def setup_rag_chain(
            self, 
            llm: Optional[ChatOpenAI] = None, 
            retriever_kwargs: Optional[Dict] = None
        ):
        """LCEL을 사용한 RAG 체인 설정"""

        # 프롬프트 템플릿 정의
        template = """당신은 한국 법률 전문가입니다. 
        주어진 컨텍스트를 기반으로 질문에 대해 정확하고 객관적으로 답변해주세요.
        관련 법 조항이나 근거가 있다면 함께 인용해주세요.

        컨텍스트: {context}
        
        질문: {question}
        
        답변:"""
        
        prompt = ChatPromptTemplate.from_template(template)
        
        # LLM 설정
        if llm is None:
            llm = ChatOpenAI(
                model_name="gpt-4.1-mini",
                temperature=0
            )


        # 벡터 스토어 검색기 설정
        if retriever_kwargs is None:
            retriever_kwargs = {"search_kwargs": {"k": 4}}

        retriever = self.vectorstore.as_retriever(**retriever_kwargs)
        
        # 문서 포맷팅 함수
        def format_documents(documents: List[Document]) -> str:
            return "\n\n".join([doc.page_content for doc in documents])

        # RAG 체인 구성
        chain = (
            {
                "context": retriever | format_documents,
                "question": RunnablePassthrough()
            }
            | prompt
            | llm
            | StrOutputParser()
        )
        
        return chain

```

</details>


In [ ]:
# RAG 시스템 초기화
rag = LegalDocumentRAG(
    collection_name="legal_docs_collection",
    persist_directory="./chroma_db"
)

# 문서 로드 및 인덱싱
docs_path = "./add_data/legal_documents"
num_chunks = rag.load_documents(docs_path)
print(f"\n총 {num_chunks}개의 문서 청크가 처리되었습니다.")

In [ ]:
# RAG 체인 설정
chain = rag.setup_rag_chain(
    llm=ChatOpenAI(
        model_name="gpt-4.1-mini",
        temperature=0
    ),
    retriever_kwargs={
        "search_type": "similarity",
        "search_kwargs": {"k": 4}
        }
)

# 질문 예시
question = "개인정보보호법에 따르면 개인정보의 수집에 대한 동의는 어떻게 해야 하나요?"

# 질문에 대한 답변 생성
answer = chain.invoke(question)

print(f"질문: {question}")
print(f"답변: {answer}")

In [ ]:
sample_questions = [
    "임대차계약에서 계약갱신요구권의 행사기간은 어떻게 되나요?",
    "개인정보 수집 시 고지해야 할 사항은 무엇인가요?",
    "근로자의 연차 유급휴가 일수는 어떻게 계산되나요?"
]

for question in sample_questions:
    answer = chain.invoke(question)
    print(f"질문: {question}")
    print(f"답변: {answer}")
    print("-" * 100)

---

## **[실습] RAG 기능 추가**

- **법적 근거 하이라이팅** 기능 추가: 마크다운 **볼드체**를 사용하여 법적 근거를 명확하게 강조

- **법률 용어 설명** 메커니즘 구현: 어려운 법률 용어를 추가로 설명하여 사용자의 이해도 향상


In [ ]:
### **[실습]**

# 여기에 코드를 작성하세요.

<details close>
<summary>💡 정답 확인</summary>

```python

from langchain_community.document_loaders import (
    UnstructuredPDFLoader,
    UnstructuredWordDocumentLoader,
)
from langchain_unstructured import UnstructuredLoader
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

import os
from typing import List, Optional, Dict, Tuple
import logging

existing_collections = {}  # 컬렉션 이름을 키로, 컬렉션 객체를 값으로 저장하는 딕셔너리

def create_new_collection(collection_name, persist_directory, embeddings, documents=None):
    """새로운 Chroma 컬렉션 생성"""
    print(f"Creating new Chroma collection: {collection_name}")
    if documents: # 문서가 제공된 경우 from_documents 사용 (load_documents 시)
        return Chroma.from_documents(
            documents=documents,
            embedding=embeddings,
            collection_name=collection_name,
            persist_directory=persist_directory
        )
    else: # 문서 없이 컬렉션만 생성 (init 시 기존 DB 로드)
        return Chroma(
            persist_directory=persist_directory,
            collection_name=collection_name,
            embedding_function=embeddings
        )


class LegalDocumentRAG:
    existing_collections = existing_collections # 클래스 변수 링크 (클래스 외부 정의 시)

    def __init__(self, collection_name: str, persist_directory: str = "./chroma_db"):
        # 로깅 설정
        logging.basicConfig(level=logging.INFO)
        self.logger = logging.getLogger(__name__)

        # 기본 설정
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
            length_function=len,
            separators=["\n\n", "\n", ".", " ", ""]
        )

        # 벡터 DB 초기화
        self.vectorstore = self.get_or_create_collection(
            self.collection_name,
            self.persist_directory,
            self.embeddings
        )


    def get_or_create_collection(self, collection_name, persist_directory, embeddings, documents=None):
        """기존 컬렉션이 있으면 반환, 없으면 새로 생성"""
        # 기존 컬렉션이 존재하는지 확인 (클래스 변수 existing_collections 사용)
        if collection_name in LegalDocumentRAG.existing_collections: # 클래스 변수 접근
            print(f"Using existing Chroma collection: {collection_name}") # 로깅 추가
            # 기존 컬렉션 반환
            return LegalDocumentRAG.existing_collections[collection_name] # 클래스 변수 접근

        # 기존 컬렉션이 없으면 새로 생성
        new_collection = create_new_collection(collection_name, persist_directory, embeddings, documents)
        LegalDocumentRAG.existing_collections[collection_name] = new_collection # 클래스 변수 저장
        return new_collection


    def _load_document(self, file_path: str) -> List[Document]:
        """파일 형식에 따라 적절한 로더를 사용하여 문서 로드"""
        self.logger.info(f"Loading document: {file_path}")

        try:
            if file_path.endswith('.pdf'):
                loader = UnstructuredPDFLoader(file_path)
            elif file_path.endswith('.docx'):
                loader = UnstructuredWordDocumentLoader(file_path)
            elif file_path.endswith('.txt'):
                loader = UnstructuredLoader(file_path)
            else:
                raise ValueError(f"Unsupported file format: {file_path}")

            documents = loader.load()

            # 메타데이터 필터링 후 파일명 추가
            for doc in documents:
                doc.metadata = {k: v for k, v in doc.metadata.items()
                                if isinstance(v, (str, int, float, bool))}
                doc.metadata["source"] = os.path.basename(file_path)

            return documents

        except Exception as e:
            self.logger.error(f"Error loading document {file_path}: {str(e)}")
            raise


    def load_documents(self, directory: str):
        """디렉토리의 모든 문서를 로드하고 인덱싱"""
        all_documents = []

        # 지원하는 파일 확장자
        supported_extensions = ('.pdf', '.docx', '.txt')

        # 디렉토리 내 모든 파일 처리
        for file in os.listdir(directory):
            if file.endswith(supported_extensions):
                file_path = os.path.join(directory, file)
                documents = self._load_document(file_path)
                all_documents.extend(documents)

        # 문서 분할
        splits = self.text_splitter.split_documents(all_documents)
        self.logger.info(f"Created {len(splits)} document chunks")

        # Chroma DB에 저장 (기존 컬렉션 재사용 또는 생성)
        self.vectorstore = self.get_or_create_collection(
            self.collection_name,
            self.persist_directory,
            self.embeddings,
            documents=splits
        )

        # persist_directory가 설정되어 있으면 자동으로 저장됨
        # self.vectorstore.persist() # Chroma 최신 버전에서 deprecated

        return len(splits)


    def setup_rag_chain(
            self,
            llm: Optional[ChatOpenAI] = None,
            retriever_kwargs: Optional[Dict] = None
        ):
        """LCEL을 사용한 RAG 체인 설정 (법적 근거 강조 및 용어 설명 포함)"""

        # --- 프롬프트 템플릿 정의 ---
        # 1. 답변 생성 프롬프트 (법적 근거 강조)
        answer_template = """당신은 한국 법률 전문가입니다.
        주어진 컨텍스트를 기반으로 질문에 대해 정확하고 객관적으로 답변해주세요.
        **답변의 근거가 되는 법 조항 또는 판례 등을 명시하고, 해당 부분을  `**볼드체**`  로 강조하여**  사용자가 명확히 알 수 있도록 해주세요.
        만약 컨텍스트에서 답을 찾을 수 없다면, '제공된 정보 내에서는 답변을 찾을 수 없습니다.' 라고 답변해주세요.

        컨텍스트: {context}

        질문: {question}

        답변:"""
        answer_prompt = ChatPromptTemplate.from_template(answer_template)

        # 2. 법률 용어 설명 프롬프트
        term_explanation_template = """주어진 텍스트에서 법률 용어를 찾아서, 각 용어에 대한 간결하고 쉬운 설명을 제공해주세요.
        설명은 일반인이 이해하기 쉬운 한국어로 작성하며, 필요한 경우 비유 또는 예시를 들어주세요.
        만약 법률 용어가 없다면, "법률 용어 설명이 필요하지 않습니다." 라고 답변해주세요.

        텍스트: {legal_answer}

        법률 용어 설명:"""
        term_explanation_prompt = ChatPromptTemplate.from_template(term_explanation_template)

        # --- LLM 설정 ---
        if llm is None:
            llm = ChatOpenAI(
                model_name="gpt-4.1-mini",
                temperature=0
            )

        # --- 벡터 스토어 검색기 설정 ---
        if retriever_kwargs is None:
            retriever_kwargs = {"search_kwargs": {"k": 4}} # top-k retrieval
        retriever = self.vectorstore.as_retriever(**retriever_kwargs)

        # --- 문서 포맷팅 함수 ---
        def format_documents(documents: List[Document]) -> str:
            return "\n\n".join([doc.page_content for doc in documents])

        # --- RAG 체인 구성 ---
        # 1. 답변 생성 체인
        answer_chain = (
            {
                "context": retriever | format_documents,
                "question": RunnablePassthrough()
            }
            | answer_prompt
            | llm
            | StrOutputParser()
        )

        # 2. 법률 용어 설명 체인
        term_explanation_chain = (
            {"legal_answer": RunnablePassthrough()}
            | term_explanation_prompt
            | llm
            | StrOutputParser()
        )  

        # 3. 최종 RAG 체인 (답변 생성 + 용어 설명) - 병렬 실행 ( RunnableParallel )
        rag_chain = RunnableParallel(
            legal_answer=answer_chain,
            term_explanation=term_explanation_chain
        )

        self.logger.info("RAG chain 설정 완료")
        return rag_chain


    def ask_question(self, question: str, rag_chain, explain_terms: bool = True) -> Tuple[str, Optional[str]]:
        """질문에 답변하고, 필요에 따라 법률 용어 설명도 제공"""
        self.logger.info(f"질문: {question}")

        if not rag_chain:
            rag_chain = self.setup_rag_chain() # rag_chain이 None일 경우 기본 체인 설정

        # RAG 체인 실행 (답변 및 용어 설명 동시 생성)
        try:
            response = rag_chain.invoke(question)
            legal_answer = response.get("legal_answer", "답변 생성 실패") # 답변
            term_explanation = response.get("term_explanation", None) # 용어 설명 (없을 수도 있음)

            self.logger.info(f"답변: {legal_answer}")
            if explain_terms:
                self.logger.info(f"법률 용어 설명: {term_explanation}")
            else:
                self.logger.info("법률 용어 설명 기능 Off")


            if explain_terms: # 법률 용어 설명 활성화 시 용어 설명도 함께 반환
                return legal_answer, term_explanation
            else: # 법률 용어 설명 비활성화 시 답변만 반환
                return legal_answer, None

        except Exception as e:
            self.logger.error(f"RAG 체인 실행 중 오류 발생: {e}")
            return "답변 생성 중 오류가 발생했습니다.", None

```

</details>


In [ ]:
# Chroma DB 설정
COLLECTION_NAME = "legal_documents"
PERSIST_DIRECTORY = "./chroma_db"

# LegalDocumentRAG 인스턴스 생성
legal_rag = LegalDocumentRAG(
    collection_name=COLLECTION_NAME,
    persist_directory=PERSIST_DIRECTORY
)

# 문서 로드 및 Chroma DB 구축 (최초 실행 시 필요)
DOCUMENTS_DIR = "./add_data/legal_documents"  # 문서가 저장된 디렉토리
if not os.path.exists(os.path.join(PERSIST_DIRECTORY, COLLECTION_NAME)): # Chroma DB가 없으면 로드 및 생성
    document_count = legal_rag.load_documents(DOCUMENTS_DIR)
    if document_count > 0:
        print(f"{document_count}개의 문서가 로드 및 Chroma DB에 저장되었습니다.")
    else:
        print("문서 로드 실패 또는 문서가 없습니다.")
else:
    print("기존 Chroma DB를 사용합니다.")

In [ ]:
# LegalDocumentRAG 인스턴스 생성
legal_rag = LegalDocumentRAG(
    collection_name=COLLECTION_NAME,
    persist_directory=PERSIST_DIRECTORY
)

# RAG 체인 설정
rag_chain = legal_rag.setup_rag_chain(
    llm=ChatOpenAI(
        model_name="gpt-4.1-mini",
        temperature=0
    ),
    retriever_kwargs={
        "search_type": "similarity",
        "search_kwargs": {"k": 4}
    }
)

# 질문 및 답변 (법률 용어 설명 활성화)
question1 = "상가건물 임대차보호법의 적용 범위는 어떻게 되나요?"
answer1, term_explanation1 = legal_rag.ask_question(question1, rag_chain, explain_terms=True)
print(f"\n질문: {question1}")
print(f"답변: {answer1}")

In [ ]:
if term_explanation1:
    print(f"법률 용어 설명: {term_explanation1}")

In [ ]:
# 질문 및 답변 (법률 용어 설명 비활성화)
question2 = "임차인이 계약 갱신을 요구할 수 있는 기간은 언제까지인가요?"
answer2, term_explanation2 = legal_rag.ask_question(question2, rag_chain, explain_terms=False)
print(f"\n질문: {question2}")
print(f"답변: {answer2}")

In [ ]:
if term_explanation2:
    print(f"법률 용어 설명: {term_explanation2}") # 비활성화 시 None 이므로 출력되지 않음